# Conditional Graph

In [56]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

In [45]:
class AgentState(TypedDict):
    number1: int
    operation: str
    number2: int
    result: int

In [46]:
def adder(state: AgentState) -> AgentState:
    """ Adds two numbers in the state and stores the result."""
    state['result'] = state['number1'] + state['number2']
    return state

def subtractor(state: AgentState) -> AgentState:
    """ Subtracts number2 from number1 in the state and stores the result."""
    state['result'] = state['number1'] - state['number2']
    return state

def multiplier(state: AgentState) -> AgentState:
    """ Multiplies two numbers in the state and stores the result."""
    state['result'] = state['number1'] * state['number2']
    return state

def divider(state: AgentState) -> AgentState:
    """ Divides number1 by number2 in the state and stores the result."""
    if state['number2'] == 0:
        raise ValueError("Cannot divide by zero.")
    state['result'] = state['number1'] / state['number2']
    return state

def decide_next_node(state: AgentState) -> AgentState:
    """ Decides the next operation based on the 'operation' field in the state."""
    operation = state['operation'].lower()
    if operation == 'add':
        return 'add_operation'
    elif operation == 'subtract':
        return 'subtract_operation'
    elif operation == 'multiply':
        return 'multiply_operation'
    elif operation == 'divide':
        return 'divide_operation'
    else:
        raise ValueError(f"Unknown operation: {operation}")

In [47]:
graph = StateGraph(AgentState)
graph.add_node('add_node', adder)
graph.add_node('subtract_node', subtractor)
graph.add_node('multiply_node', multiplier)
graph.add_node('divide_node', divider)

graph.add_node('decision_node', lambda state:state)

graph.add_edge(START, 'decision_node')
graph.add_conditional_edges(
    'decision_node',
    decide_next_node,
    {
        'add_operation': 'add_node',
        'subtract_operation': 'subtract_node',
        'multiply_operation': 'multiply_node',
        'divide_operation': 'divide_node'
    }
)

graph.add_edge('add_node', END)
graph.add_edge('subtract_node', END)
graph.add_edge('multiply_node', END)
graph.add_edge('divide_node', END)

In [48]:
app = graph.compile()

In [55]:
initial_state = AgentState(
    number1 = 10,
    operation= 'subtract',
    number2 = 5,
    result=0
)
print(app.invoke(initial_state))
print("Result: ", app.invoke(initial_state)['result'])

{'number1': 10, 'operation': 'subtract', 'number2': 5, 'result': 5}
Result:  5
